1. Machine Learning kya hai: Traditional programming mein tum rules likhte ho (if Recency > 90: churned). ML mein tum data aur sahi answers dete ho, aur algorithm khud rules seekhta hai.

2. Supervised Learning: Training data mein sahi answer (label) pehle se maujood hota hai. Do main types:

Classification: answer ek category hai (Churned: Yes/No). Yeh tumhara project hai.
Regression: answer ek number hai (jaise next month ka spend). Day 75 ka RMSE isi type ke liye hai.

(Unsupervised learning mein labels nahi hote, jaise customer clustering. Abhi ke liye sirf naam jaan lo.)

3. Zaroori shabdavali:

Features (X): input columns jinse model seekhta hai
Target (y): jo predict karna hai (Churned)
Training set: jis data se model seekhta hai
Test set: jis data par model ko pehli baar "exam" dete hain
Overfitting: model training data ko ratt leta hai, naye data par fail ho jaata hai

4. Ek toy example se poora workflow dekho (chhota synthetic data, taaki sirf workflow samajh aaye):

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(1)
n = 300
toy = pd.DataFrame({
    'study_hours': np.random.uniform(0, 10, n),
    'attendance': np.random.uniform(40, 100, n)
})
# Pass hone ki probability study_hours aur attendance par depend karti hai
prob = 1 / (1 + np.exp(-(0.6 * toy['study_hours'] + 0.04 * toy['attendance'] - 5)))
toy['passed'] = (np.random.rand(n) < prob).astype(int)

X = toy[['study_hours', 'attendance']]
y = toy['passed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)            # training
preds = model.predict(X_test)          # prediction

print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy :", model.score(X_test, y_test))

Train accuracy: 0.8416666666666667
Test accuracy : 0.7666666666666667


yeh 4 steps (split, fit, predict, score) har ML project ka skeleton hain. Day 63-67 mein inhe apne churn data par karenge.

5. Overfitting ka demo (Day 69 ke Decision Tree ke liye intuition):

In [4]:
from sklearn.tree import DecisionTreeClassifier

for depth in [1, 3, 5, 10, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f"max_depth={depth}: train={tree.score(X_train, y_train):.2f}, test={tree.score(X_test, y_test):.2f}")

max_depth=1: train=0.85, test=0.73
max_depth=3: train=0.90, test=0.68
max_depth=5: train=0.93, test=0.68
max_depth=10: train=1.00, test=0.63
max_depth=None: train=1.00, test=0.63


Dekho ki depth badhne par train accuracy badhti rehti hai, lekin test accuracy ruk jaati hai ya girti hai. Train aur test ka yeh gap hi overfitting hai.

6. Apne project ka problem statement likho (markdown cell mein, ise README mein bhi daalna):

## ML Problem Statement
- **Type:** Binary classification (supervised)
- **Target (y):** Churned (1 = churned, 0 = active)
- **Features (X):** customer behaviour columns (Frequency, Monetary, country, etc.)
- **Business goal:** churn hone se pehle at-risk customers pehchaanna, taaki retention offer diya ja sake
- **Success metric:** Accuracy nahi (class imbalance, Day 56B). Recall, F1 aur ROC-AUC (Day 71-74)

7. Aage ka ek zaroori warning (Day 62 mein hum isse solve karenge):

Week 12 ke end mein maine data leakage ka zikr kiya tha. Abhi sirf concept samjho:

Tumhara Churned label Recency se bana hai (Recency > 90 din).
Agar Recency ko feature rakhoge, model sirf wahi rule dobara seekh lega aur ~100% accuracy dikhayega.
Woh asli prediction nahi hogi. Model ko churn pehle se predict karna hai, churn ki definition dobara nahi seekhni.

Is liye Day 62 mein hum features ko aise banayenge ki model sirf past behaviour dekhe aur future churn predict kare.

practice questions:

1. Toy example mein test_size ko 0.2 se 0.5 karo aur dekho test accuracy kitni hilti hai.

Jab hum test_size=0.2 rakhte hain, toh model ko 80% data training ke liye milta hai aur 20% data test ke liye.

Jab tum ise badhakar test_size=0.5 karte ho, toh train aur test data 50-50% split ho jaate hain.

- Iska asar seedha model ki learning aur accuracy par padta hai:

    - Training Data Kam Ho Gaya: Model ke paas sikhne ke liye ab sirf 150 rows bachi hain (300 ki jagah), jis wajah se ho sakta hai wo patterns ko utne acche se na pakad paaye.

    - Test Accuracy Ka Hilna: Jab humne isse run karke dekha, toh test accuracy 76.67% se badhakar 80.67% ho gayi. Yeh thoda counter-intuitive lag sakta hai ki training data kam hone par accuracy badh kaise gayi, par yeh data variance ki wajah se hota hai. Chote datasets mein kaunsa data train mein gaya aur kaunsa test mein, iska accuracy par bada asar padta hai. Asli projects mein (jahan hazaron/lakhon rows hoti hain), kam training data se model aksar kamzor ho jaata hai.

2. Apne shabdon mein likho: classification aur regression ka ek-ek example apne churn project ke context mein do.

Apne RFM / Customer Churn project ke hisaab se dono ka fark bilkul saaf hai:

- Classification Example:

    - Kya predict karna hai: Kya yeh customer agle mahine churn karega ya nahi? (0 matlab active, 1 matlab churned).
    - Kyun: Kyunki iska answer ek category/class hai. Humein binary decision lena hai taaki hum at-risk customers ko pehchaan kar unhe retention discount ya offer bhej sakein.


- Regression Example:

    - Kya predict karna hai: Agle 3 mahino mein yeh customer total kitna paisa kharch karega (Customer Lifetime Value ya Monetary Spend ka exact amount, jaise ₹4,500 ya ₹12,200)?
    - Kyun: Kyunki yahan answer koi category nahi balki ek continuous number (floating point value) hai.